In [1]:
import numpy as np
import pandas as pd

from datetime import datetime
from scipy.stats import skew 
from scipy.special import boxcox1p
from scipy.stats import boxcox_normmax
from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error
from mlxtend.regressor import StackingCVRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import matplotlib.pyplot as plt
import scipy.stats as stats
import sklearn.linear_model as linear_model
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.neighbors import KDTree
from sklearn.decomposition import TruncatedSVD
from sklearn.neighbors import BallTree
from sklearn.model_selection import train_test_split
import lightgbm as lgb
import os
print(os.listdir())
import category_encoders as ce

import warnings
warnings.filterwarnings('ignore')

['house_price_competition_day_2_failed_trying.ipynb', 'house_price_advanced_view.ipynb', 'addr_kmeans.pkl', 'submission.csv', 'house_price_advanced_models.ipynb', 'my_model_submission.csv1', 'submission5.csv', 'my_model_submission4.csv', 'house_price', 'addr_umap.pkl', 'Day1.ipynb', 'titanic', 'house-prices-advanced-regression-techniques.zip', 'titanic.zip', 'my_model_submission3.csv', 'house_price_competition_view_day2.ipynb', 'my_model_submission0.csv', 'X_umap.npy', 'addr_tfidf.pkl', 'Day2 Housing Price.ipynb', 'pca_model.pkl', 'my_model_submission.csv', 'predictions.csv', 'umap_model.pkl', 'predictions4.csv', 'my_model_submission1.csv', 'competition_day7.ipynb', 'house_price_view.ipynb', 'home-data-for-ml-course.zip', 'Untitled2.ipynb', '.ipynb_checkpoints', 'home-data-for-ml-course', 'house_price.ipynb', 'predictions3.csv', 'failed_trying_with_nns_day_9.ipynb', 'not_so_bad_trying_with_lightgbm_day_8.ipynb', 'my_model_submission2.csv', 'best_interval_model.pth', 'predictions2.csv']

In [ ]:
train = pd.read_csv('house_price/dataset.csv')
test = pd.read_csv('house_price/test.csv')
print ("Data is loaded!")

quantitative = [f for f in train.columns if train.dtypes[f] != 'object']
quantitative.remove('sale_price')
quantitative.remove('id')
qualitative = [f for f in train.columns if train.dtypes[f] == 'object']

sns.set_style("whitegrid")
missing = train.isnull().sum()
missing = missing[missing > 0]
print(missing)

sns.set_style("whitegrid")
missing = test.isnull().sum()
missing = missing[missing > 0]
print(missing)

def dataset_fill_null(obj):
    obj['subdivision'].fillna('Unknown', inplace=True)
    obj.drop(columns=['sale_nbr'], inplace=True)
    obj['submarket'].fillna('Unknown', inplace=True)


dataset_fill_null(train)
dataset_fill_null(test)
print(train.shape)
print(test.shape)

# 构造原始地址字段
train_ID = train['id']
test_ID = test['id']
# Now drop the  'Id' colum since it's unnecessary for  the prediction process.
drop_cols=['id',#row_id,没有任何信息.
           'golf',#20万数据 198756都是0,基本没什么信息了.
           'view_rainier',#20万数据,198588都是0.
           'view_skyline',#20万数据,198517都是0.
           'view_lakesamm',#20万数据,198776都是0.
           'view_otherwater',#20万数据,198473都是0.
           'view_other',#20万数据,198833都是0.
          ]
train.drop(drop_cols, axis=1, inplace=True)
test_raw = test.drop(drop_cols, axis=1, inplace=True)

# Deleting outliers
train.reset_index(drop=True, inplace=True)
# We use the numpy fuction log1p which  applies log(1+x) to all elements of the column
# train["sale_price"] = np.log1p(train["sale_price"])
y = train.sale_price.reset_index(drop=True)

In [9]:
def preprocess_and_encode00(df, y=None, drop_high_card=True):
    df = df.copy()
    
    # ================= 清洗阶段 ================= #
    addresses = (df['city'].fillna('') + ' ' + df['subdivision'].fillna('')).str.lower()
    
    # 分词（空格切）
    # 用 CountVectorizer 可以保留频率稀疏性
    
    vectorizer = CountVectorizer(
        max_features=1000,         # 可调大小
        stop_words=None,      # 去常见词
        token_pattern=r'\b\w+\b',  # 标准单词
        ngram_range=(1, 2)         # 一元和二元组都试试
    )
    address_vecs = vectorizer.fit_transform(addresses)
    
    address_df = pd.DataFrame(address_vecs.toarray(), columns=vectorizer.get_feature_names_out())
    address_df.index = df.index
    df = pd.concat([df, address_df], axis=1)
    df = df.drop(columns=['city', 'subdivision'])

    # 使用同一个 address_vecs（CountVectorizer 输出）
    svd = TruncatedSVD(n_components=50, random_state=42)
    address_pca = svd.fit_transform(address_vecs)
    
    # 转成 DataFrame 并拼接
    pca_df = pd.DataFrame(address_pca, columns=[f'address_pca_{i+1}' for i in range(address_pca.shape[1])], index=df.index)
    df = pd.concat([df, pca_df], axis=1)

    # 补充缺失类别为 'None'
    cat_cols = df.select_dtypes(include=['object', 'category', 'string']).columns
    
    for col in cat_cols:
        df[col] = df[col].fillna('None')

    # 填充数值缺失为 0
    num_cols = df.select_dtypes(include=[np.number]).columns
    for col in num_cols:
        df[col] = df[col].fillna(0)

    # 拆解日期特征
    if 'sale_date' in df.columns:
        df['sale_year'] = pd.to_datetime(df['sale_date']).dt.year
        df['sale_month'] = pd.to_datetime(df['sale_date']).dt.month
        df = df.drop(columns='sale_date')

        df['house_age'] = df['sale_year'] - df['year_built']
        df['reno_age'] = df['sale_year'] - df['year_reno']
        df['has_reno'] = (df['year_reno'] > 0).astype(int)
        df['land_imp_ratio'] = df['land_val'] / (df['imp_val'] + 1e-5)


    # ================= 编码阶段 ================= #
    cat_cols = df.select_dtypes(include='object').columns
    low_card_cols = []
    high_card_cols = []
    
    for col in cat_cols:
        try:
            n_unique = df[col].nunique()
            if isinstance(n_unique, (int, np.integer)):
                if n_unique <= 50:
                    low_card_cols.append(col)
                else:
                    high_card_cols.append(col)
            else:
                print(f"[跳过] {col} 的 nunique 结果不是标量: {n_unique}")
        except Exception as e:
            print(f"[异常] {col}: {e}")


    # One-hot 编码
    X_cat = pd.get_dummies(df[low_card_cols], dummy_na=True)

    # Target 编码
    if y is not None and high_card_cols:
        encoder = ce.TargetEncoder()
        X_target = encoder.fit_transform(df[high_card_cols], y)
        X_target.columns = [f"{col}_te" for col in high_card_cols]
    else:
        X_target = pd.DataFrame(index=df.index)

    # 数值标准化
    num_cols = df.select_dtypes(include=[np.number]).columns
    scaler = StandardScaler()
    X_num_scaled = pd.DataFrame(scaler.fit_transform(df[num_cols]), columns=num_cols, index=df.index)

    # 拼接所有
    X_final_df = pd.concat([X_num_scaled, X_cat, X_target], axis=1)

    return X_final_df


In [ ]:
def preprocess_and_encode(df, y=None, encoder_bundle=None, drop_high_card=True):
    df = df.copy()
    
    # ================= 清洗阶段 ================= #
    if encoder_bundle is None:
        # 训练阶段
        address_text = (df['city'].fillna('') + ' ' + df['subdivision'].fillna('')).str.lower()
        vectorizer = CountVectorizer(max_features=1000, token_pattern=r'\b\w+\b', ngram_range=(1, 2))
        address_vec = vectorizer.fit_transform(address_text)
        svd = TruncatedSVD(n_components=50, random_state=42)
        address_pca = svd.fit_transform(address_vec)
    else:
        # 推理阶段
        vectorizer = encoder_bundle['vectorizer']
        svd = encoder_bundle['svd']
        address_text = (df['city'].fillna('') + ' ' + df['subdivision'].fillna('')).str.lower()
        address_vec = vectorizer.transform(address_text)
        address_pca = svd.transform(address_vec)

    address_df = pd.DataFrame(address_pca, columns=[f'address_pca_{i+1}' for i in range(address_pca.shape[1])], index=df.index)
    df = df.drop(columns=['city', 'subdivision'], errors='ignore')
    df = pd.concat([df, address_df], axis=1)

    # 补充缺失
    cat_cols = df.select_dtypes(include=['object', 'category', 'string']).columns
    df[cat_cols] = df[cat_cols].fillna('None')
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = df[num_cols].fillna(0)

    # 日期和派生
    if 'sale_date' in df.columns:
        df['sale_year'] = pd.to_datetime(df['sale_date']).dt.year
        df['sale_month'] = pd.to_datetime(df['sale_date']).dt.month
        df = df.drop(columns='sale_date')
        df['house_age'] = df['sale_year'] - df['year_built']
        df['reno_age'] = df['sale_year'] - df['year_reno']
        df['has_reno'] = (df['year_reno'] > 0).astype(int)
        df['land_imp_ratio'] = df['land_val'] / (df['imp_val'] + 1e-5)

    # 编码分类变量
    cat_cols = df.select_dtypes(include='object').columns
    low_card_cols = []
    high_card_cols = []
    
    for col in cat_cols:
        try:
            n_unique = df[col].nunique()
            if isinstance(n_unique, (int, np.integer)):
                if n_unique <= 50:
                    low_card_cols.append(col)
                else:
                    high_card_cols.append(col)
            else:
                print(f"[跳过] {col} 的 nunique 结果不是标量: {n_unique}")
        except Exception as e:
            print(f"[异常] {col}: {e}")

    X_cat = pd.get_dummies(df[low_card_cols], dummy_na=True)

    if encoder_bundle is None:
        target_encoder = ce.TargetEncoder()
        X_target = target_encoder.fit_transform(df[high_card_cols], y) if y is not None else pd.DataFrame(index=df.index)
    else:
        target_encoder = encoder_bundle['target_encoder']
        X_target = target_encoder.transform(df[high_card_cols]) if high_card_cols else pd.DataFrame(index=df.index)
    X_target.columns = [f"{col}_te" for col in high_card_cols]

    # 数值标准化
    num_cols = df.select_dtypes(include=[np.number]).columns
    if encoder_bundle is None:
        scaler = StandardScaler()
        X_num_scaled = pd.DataFrame(scaler.fit_transform(df[num_cols]), columns=num_cols, index=df.index)
    else:
        scaler = encoder_bundle['scaler']
        X_num_scaled = pd.DataFrame(scaler.transform(df[num_cols]), columns=num_cols, index=df.index)

    X_final_df = pd.concat([X_num_scaled, X_cat, X_target], axis=1)

    if encoder_bundle is None:
        encoder_bundle = {
            'vectorizer': vectorizer,
            'svd': svd,
            'target_encoder': target_encoder,
            'scaler': scaler
        }

    return X_final_df, encoder_bundle



def add_knn_price_features(df, base_df=None, lat_col='latitude', lon_col='longitude',
                           target_col='target', ks=[5, 10, 20]):
    if base_df is None:
        base_df = df  # 默认自己为近邻池

    coords_query = df[[lat_col, lon_col]].values
    coords_base = base_df[[lat_col, lon_col]].values
    tree = KDTree(coords_base, metric='euclidean')

    base_targets = base_df[target_col].values
    knn_features = {}

    for k in ks:
        dists, indices = tree.query(coords_query, k=k)
        neighbor_targets = base_targets[indices]

        knn_features[f'knn_price_mean_{k}'] = neighbor_targets.mean(axis=1)
        knn_features[f'knn_price_std_{k}'] = neighbor_targets.std(axis=1)
        knn_features[f'knn_price_range_{k}'] = neighbor_targets.max(axis=1) - neighbor_targets.min(axis=1)

    for col, val in knn_features.items():
        df[col] = val

    return df


def add_radius_knn_features(df, base_df, lat_col='latitude', lon_col='longitude', target_col='target', radius=0.01, max_neighbors=100):
    coords = np.radians(base_df[[lat_col, lon_col]])
    tree = BallTree(coords, metric='haversine')  # 地理距离计算更准

    df_coords = np.radians(df[[lat_col, lon_col]])
    indices = tree.query_radius(df_coords, r=radius)

    # 每个样本找到若干邻居索引后，聚合
    agg_means, agg_stds, agg_counts = [], [], []
    base_targets = base_df[target_col].values

    for idxs in indices:
        if len(idxs) > 1:
            if len(idxs) > max_neighbors:
                idxs = idxs[:max_neighbors]
            neigh_vals = base_targets[idxs]
            agg_means.append(np.mean(neigh_vals))
            agg_stds.append(np.std(neigh_vals))
            agg_counts.append(len(idxs))
        else:
            agg_means.append(np.nan)
            agg_stds.append(np.nan)
            agg_counts.append(0)

    df['radius_knn_mean'] = agg_means
    df['radius_knn_std'] = agg_stds
    df['radius_knn_count'] = agg_counts

    return df


def add_knn_price_features_radius(df, base_df, lat_col='latitude', lon_col='longitude',
                                   target_col='sale_price', radius=0.01, max_neighbors=100):
    df = df.copy()
    
    coords_query = df[[lat_col, lon_col]].astype(np.float32).values
    coords_base = base_df[[lat_col, lon_col]].astype(np.float32).values
    targets_base = base_df[target_col].astype(np.float32).values

    tree = KDTree(coords_base, leaf_size=40, metric='euclidean')
    neighbor_indices = tree.query_radius(coords_query, r=radius)

    means, stds, ranges = [], [], []

    for i, inds in enumerate(neighbor_indices):
        # 排除自身（仅当 base_df 是 df）
        if base_df is df:
            inds = inds[inds != i]
        
        if len(inds) == 0:
            means.append(np.nan)
            stds.append(np.nan)
            ranges.append(np.nan)
        else:
            if len(inds) > max_neighbors:
                inds = inds[:max_neighbors]
            vals = targets_base[inds]
            means.append(np.mean(vals))
            stds.append(np.std(vals))
            ranges.append(np.max(vals) - np.min(vals))

    df[f'knn_radius_mean'] = means
    df[f'knn_radius_std'] = stds
    df[f'knn_radius_range'] = ranges

    return df

# 应用预处理
# 对训练集（自己做自己）

X_train_raw, X_val, y_train, y_val = train_test_split(
    train, y, test_size=0.2, random_state=42
)

test_raw = test.copy()  # 或者正确读取原始测试集

X_train = add_knn_price_features(X_train_raw, base_df=X_train_raw, target_col='sale_price')
X_val = add_knn_price_features(X_val, base_df=X_train_raw, target_col='sale_price')
X_train_full = add_knn_price_features(train, base_df=train, target_col='sale_price')
# 对测试集（用训练集做 base）
test = add_knn_price_features(test_raw, base_df=X_train_raw, target_col='sale_price')

X_train = add_knn_price_features_radius(X_train, base_df=X_train_raw, target_col='sale_price')
X_val = add_knn_price_features_radius(X_val, base_df=X_train_raw, target_col='sale_price')
X_train_full = add_knn_price_features_radius(X_train_full, base_df=train, target_col='sale_price')
# 对测试集（用训练集做 base）
test = add_knn_price_features_radius(test, base_df=X_train_raw, target_col='sale_price')

X_train = X_train.drop(['sale_price'], axis=1)
X_val = X_val.drop(['sale_price'], axis=1)
X_train_full = X_train_full.drop(['sale_price'], axis=1)
X_train, encoder_bundle = preprocess_and_encode(X_train, y_train)
X_val, _ = preprocess_and_encode(X_val, encoder_bundle=encoder_bundle)
test, _ = preprocess_and_encode(test, encoder_bundle=encoder_bundle)
X_train_full, _ = preprocess_and_encode(X_train_full, y)

In [13]:
print('X_train', X_train.shape, 'y_train', y_train.shape, 'X_val', X_val.shape, 'test', test.shape, 'X_train_full', X_train_full.shape, 'y', y.shape)

X_train (160000, 131) y_train (160000,) X_val (40000, 131) test (200000, 131) X_train_full (200000, 131) y (200000,)


In [22]:
def get_lgb_quantile_model(alpha):
    return lgb.LGBMRegressor(
        objective='quantile',
        alpha=alpha,
        n_jobs=-1,
        num_leaves=4,
        learning_rate=0.01,
        n_estimators=5000,
        max_bin=200,
        bagging_fraction=0.75,
        bagging_freq=5,
        bagging_seed=7,
        feature_fraction=0.2,
        feature_fraction_seed=7,
        min_data_in_leaf=20,   # 调大一点防过拟合 tail
        verbose=-1
    )


In [15]:
def train_quantile_model(X, y, quantile):
    params = {
        'objective': 'quantile',
        'alpha': quantile,
        'n_estimators': 1000,
        'learning_rate': 0.05,
        'min_data_in_leaf': 30,
        'verbosity': -1
    }
    model = lgb.LGBMRegressor(**params)
    model.fit(X, y)
    return model


In [16]:
def interval_score(y_true, lower, upper, alpha=0.1):
    """
    Compute Wα for a given prediction interval [lower, upper] and true values y.
    """
    interval_width = upper - lower
    below = y_true < lower
    above = y_true > upper
    inside = (lower <= y_true) & (y_true <= upper)
    
    penalty = np.zeros_like(y_true, dtype=float)
    penalty[below] = (2 / alpha) * (lower[below] - y_true[below])
    penalty[above] = (2 / alpha) * (y_true[above] - upper[above])

    return np.mean(interval_width + penalty)


In [25]:


model_lower = get_lgb_quantile_model(alpha=0.05)
model_upper = get_lgb_quantile_model(alpha=0.95)

model_lower.fit(X_train, y_train)
model_upper.fit(X_train, y_train)

pred_lower = model_lower.predict(X_val)
pred_upper = model_upper.predict(X_val)

score = interval_score(y_val, pred_lower, pred_upper, alpha=0.1)
print(f"W_alpha score: {score:.5f}")

W_alpha score: 0.84880


In [17]:
model_lower = train_quantile_model(X_train, y_train, quantile=0.05)
model_upper = train_quantile_model(X_train, y_train, quantile=0.95)

pred_lower = model_lower.predict(X_val)
pred_upper = model_upper.predict(X_val)

score = interval_score(y_val, pred_lower, pred_upper, alpha=0.1)
print(f"W_alpha score: {score:.5f}")

W_alpha score: 877480.68134


In [ ]:
model_lower = train_quantile_model(X, y, quantile=0.05)
model_upper = train_quantile_model(X, y, quantile=0.95)




In [ ]:
pred_lower = model_lower.predict(X_sub)
pred_upper = model_upper.predict(X_sub)



In [ ]:
pred_lower = np.expm1(pred_lower)
pred_upper = np.expm1(pred_upper)

submission = pd.DataFrame({
    'id': test_ID,  # test 原始数据中的 ID
    'pi_lower': pred_lower,  # 下限预测
    'pi_upper': pred_upper   # 上限预测
})
submission.to_csv('submission.csv', index=False)

In [ ]:
def best_features(model):
    # 获取特征重要性
    importance = model.feature_importances_
    features = X_train.columns
    
    # 打包成 DataFrame
    feat_imp = pd.DataFrame({
        'feature': features,
        'importance': importance
    }).sort_values('importance', ascending=False)
    
    # 显示前 30 个最重要的特征
    print(feat_imp.head(50))


best_features(model_lower)
best_features(model_upper)

In [18]:
def winkler_score(y_true, lower, upper, alpha=0.1, return_coverage=False):

    y_true = np.asarray(y_true)
    lower = np.asarray(lower)
    upper = np.asarray(upper)

    width = upper - lower
    penalty_lower = 2 / alpha * (lower - y_true)
    penalty_upper = 2 / alpha * (y_true - upper)

    score = width.copy()
    score += np.where(y_true < lower, penalty_lower, 0)
    score += np.where(y_true > upper, penalty_upper, 0)

    if return_coverage:
        inside = (y_true >= lower) & (y_true <= upper)
        coverage = np.mean(inside)
        return np.mean(score), coverage

    return np.mean(score)

In [19]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_pinball_loss, make_scorer
import lightgbm as lgb

In [ ]:


param_grid = {
    'num_leaves': [4, 8, 16],
    'learning_rate': [0.01, 0.05],
    'n_estimators': [1000, 2000],
    'feature_fraction': [0.6, 0.8],
    'bagging_fraction': [0.6, 0.8],
    'min_data_in_leaf': [20, 30],
}

def pinball_scorer(y_true, y_pred):
    return -mean_pinball_loss(y_true, y_pred, alpha=0.05)

scorer = make_scorer(pinball_scorer, greater_is_better=True)


model = lgb.LGBMRegressor(objective='quantile', alpha=0.05)
grid = GridSearchCV(model, param_grid, cv=3, scoring=scorer)
grid.fit(X, y)

print(grid.best_params_)


In [ ]:
params = {
    'objective': 'quantile',
    'alpha': 0.05,  # 下限模型
    'num_leaves': 16,
    'learning_rate': 0.05,
    'n_estimators': 2000,
    'feature_fraction': 0.6,
    'bagging_fraction': 0.6,
    'min_data_in_leaf': 30,
    'verbosity': -1
}

model_lower = lgb.LGBMRegressor(**params)
model_lower.fit(X_train, y_train)

# 上限只改 alpha
params['alpha'] = 0.95
model_upper = lgb.LGBMRegressor(**params)
model_upper.fit(X_train, y_train)


In [18]:
print('X_train', X_train.shape, 'y_train', y_train.shape, 'X_val', X_val.shape, 'y_val:', y_val, 'test', test.shape)

X_train (160000, 131) y_train (160000,) X_val (40000, 131) y_val: 119737    12.448709
72272     12.899222
158154    13.933294
65426     12.793862
30074     13.665851
            ...    
4174      13.068965
91537     13.120363
156449    12.886644
184376    13.592368
6584      13.011434
Name: sale_price, Length: 40000, dtype: float64 test (200000, 131)


In [20]:
def try_param_set(X, y, param_overrides, alpha):
    base_params = {
        'objective': 'quantile',
        'alpha': alpha,
        'reg_alpha': 1.0,
        'reg_lambda': 1.0,
        'n_estimators': 1000,
        'learning_rate': 0.05,
        'min_data_in_leaf': 30,
        'verbosity': -1
    }
    base_params.update(param_overrides)
    model = lgb.LGBMRegressor(**base_params)
    model.fit(X, y)
    return model, param_overrides


In [21]:
num_leaves = [5, 6, 7, 8, 9, 10]
learning_rate = [0.01, 0.02, 0.03, 0.05]

for leaves in num_leaves:
    for lr in learning_rate:
        lower_model, params = try_param_set(X_train, y_train, {'num_leaves': leaves, 'learning_rate': lr}, 0.05)
        upper_model, params = try_param_set(X_train, y_train, {'num_leaves': leaves, 'learning_rate': lr}, 0.95)
        
        pred_lower = lower_model.predict(X_val)
        pred_upper = upper_model.predict(X_val)
        score, coverage = winkler_score(y_val, pred_lower, pred_upper, 0.1, True)
        print(score, coverage, params)


1027500.3438770392 0.800525 {'num_leaves': 5, 'learning_rate': 0.01}
932404.582971704 0.788825 {'num_leaves': 5, 'learning_rate': 0.02}
889210.6716976934 0.786975 {'num_leaves': 5, 'learning_rate': 0.03}
842091.528592703 0.786425 {'num_leaves': 5, 'learning_rate': 0.05}
1009406.9459216971 0.798525 {'num_leaves': 6, 'learning_rate': 0.01}
921743.0843413486 0.786725 {'num_leaves': 6, 'learning_rate': 0.02}
881527.2416224029 0.785275 {'num_leaves': 6, 'learning_rate': 0.03}
828212.1475793752 0.78415 {'num_leaves': 6, 'learning_rate': 0.05}
996625.0486526188 0.797 {'num_leaves': 7, 'learning_rate': 0.01}
908746.4292943642 0.7835 {'num_leaves': 7, 'learning_rate': 0.02}
876081.0876667338 0.783625 {'num_leaves': 7, 'learning_rate': 0.03}
833195.3901016694 0.78095 {'num_leaves': 7, 'learning_rate': 0.05}
992384.3106420279 0.794475 {'num_leaves': 8, 'learning_rate': 0.01}
908603.1408091545 0.783475 {'num_leaves': 8, 'learning_rate': 0.02}
870926.1014430473 0.781375 {'num_leaves': 8, 'learning_

In [95]:
lower_model, params = try_param_set(X_train, y_train, {'num_leaves': 8, 'learning_rate': 0.05}, 0.05)
upper_model, params = try_param_set(X_train, y_train, {'num_leaves': 8, 'learning_rate': 0.05}, 0.95)

pred_lower = lower_model.predict(X_val)
pred_upper = upper_model.predict(X_val)


In [96]:
score, coverage = winkler_score(y_val, pred_lower, pred_upper, 0.1, True)
print(score, coverage, params)

0.8551444621422026 0.7941 {'num_leaves': 8, 'learning_rate': 0.05}


In [98]:
pred_lower = lower_model.predict(test)
pred_upper = upper_model.predict(test)
submission = pd.DataFrame({
    'id': test_ID,
    'pi_lower': np.expm1(pred_lower),
    'pi_upper': np.expm1(pred_upper)
})
submission.to_csv('submission5.csv', index=False)

In [ ]:
import optuna
from sklearn.model_selection import train_test_split
import lightgbm as lgb
from sklearn.metrics import mean_pinball_loss
import numpy as np

def winkler_score(y_true, lower, upper, alpha=0.1):
    y_true = np.asarray(y_true)
    lower = np.asarray(lower)
    upper = np.asarray(upper)
    width = upper - lower
    penalty_lower = 2 / alpha * (lower - y_true)
    penalty_upper = 2 / alpha * (y_true - upper)
    score = width.copy()
    score += np.where(y_true < lower, penalty_lower, 0)
    score += np.where(y_true > upper, penalty_upper, 0)
    return np.mean(score)

# ========= 数据划分 =========
X_tr, X_val_, y_tr, y_val_ = X_train, X_val, y_train, y_val

def train_quantile_model(X, y, alpha, params):
    params = params.copy()
    params.update({
        'objective': 'quantile',
        'alpha': alpha,
        'verbosity': -1
    })
    model = lgb.LGBMRegressor(**params)
    model.fit(X, y,
              eval_set=[(X_val_, y_val_)],
               callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(0)]
                )
    return model

def objective(trial):
    common_params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 3000, step=500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 5, 64),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 10, 100, step=10),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'lambda_l1': trial.suggest_float('lambda_l1', 0.0, 5.0),
        'lambda_l2': trial.suggest_float('lambda_l2', 0.0, 5.0),
    }

    lower_model = train_quantile_model(X_tr, y_tr, alpha=0.05, params=common_params)
    upper_model = train_quantile_model(X_tr, y_tr, alpha=0.95, params=common_params)

    pred_lower = lower_model.predict(X_val_)
    pred_upper = upper_model.predict(X_val_)

    score = winkler_score(y_val_, pred_lower, pred_upper, alpha=0.1)
    return score  # 越小越好

# ========= 启动搜索 =========
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=100, n_jobs=4)
print("Best params:", study.best_params)



In [57]:
params = {'n_estimators': 3000, 'learning_rate': 0.0143650970821106, 'num_leaves': 6, 'min_data_in_leaf': 60, 'feature_fraction': 0.9187547403541628, 'bagging_fraction': 0.8078659488207384, 'bagging_freq': 3, 'lambda_l1': 4.242553703940317, 'lambda_l2': 0.6129405566341026}
params.update({'objective': 'quantile','verbosity': -1})

In [58]:
import numpy as np
from sklearn.metrics import mean_pinball_loss

def optimize_interval_scaling(y_true, pred_low, pred_high, alpha=0.1, step=0.01):
    best_score = float('inf')
    best_params = None

    for scale_low in np.arange(0.9, 1.01, step):         # 0.9 ~ 1.0，略微向中心收缩
        for scale_high in np.arange(0.9, 1.01, step):
            # 中心点和半宽度
            center = (pred_low + pred_high) / 2
            half_width = (pred_high - pred_low) / 2

            # 收缩区间
            low_adj = center - half_width * scale_low
            high_adj = center + half_width * scale_high

            # 覆盖率计算
            coverage = np.mean((y_true >= low_adj) & (y_true <= high_adj))

            if coverage >= 1 - alpha:  # 覆盖率达标
                width = high_adj - low_adj
                under = (y_true < low_adj).astype(float)
                over = (y_true > high_adj).astype(float)
                penalty = (2 / alpha) * ((low_adj - y_true) * under + (y_true - high_adj) * over)
                score = np.mean(width + penalty)

                if score < best_score:
                    best_score = score
                    best_params = (scale_low, scale_high)

    if best_params is None:
        print("[警告] 没有任何区间满足覆盖率 >= 1 - alpha 的要求。返回默认缩放 (1.0, 1.0)")
        best_score = float('inf')
        best_params = (1.0, 1.0)

    return best_score, best_params


In [59]:
lower_model1 = lgb.LGBMRegressor(**params, alpha=0.05, n_jobs=-1)
upper_model1 = lgb.LGBMRegressor(**params, alpha=0.95, n_jobs=-1)
lower_model1.fit(X_train, y_train)
upper_model1.fit(X_train, y_train)
pred_lower = lower_model.predict(X_val)
pred_upper = upper_model.predict(X_val)
print("Finished")

Finished


In [60]:
score, (best_scale_low, best_scale_high) = optimize_interval_scaling(
    y_val, pred_lower, pred_upper, alpha=0.1
)
print(f"Best score: {score:.4f}, scale_low: {best_scale_low}, scale_high: {best_scale_high}")


[警告] 没有任何区间满足覆盖率 >= 1 - alpha 的要求。返回默认缩放 (1.0, 1.0)
Best score: inf, scale_low: 1.0, scale_high: 1.0


In [61]:
coverage = np.mean((y_val >= pred_lower) & (y_val <= pred_upper))
avg_width = np.mean(pred_upper - pred_lower)

print(f"原始覆盖率: {coverage:.4f}")
print(f"平均区间宽度: {avg_width:.4f}")

原始覆盖率: 0.8232
平均区间宽度: 0.4236


In [62]:
lower_model = lgb.LGBMRegressor(**params, alpha=0.05, n_jobs=-1)
upper_model = lgb.LGBMRegressor(**params, alpha=0.95, n_jobs=-1)
lower_model.fit(X_train_full, y)
upper_model.fit(X_train_full, y)
pred_lower = lower_model.predict(test)
pred_upper = upper_model.predict(test)
print("Finished")

Finished


In [63]:
submission = pd.DataFrame({
    'id': test_ID,
    'pi_lower': np.expm1(pred_lower),
    'pi_upper': np.expm1(pred_upper)
})
submission.to_csv('submission5.csv', index=False)

In [ ]:
other_params = {
    'n_estimators': 1000,
    'learning_rate': 0.05,
    'max_depth': 4,
    'max_features': 'sqrt',
    'min_samples_leaf': 15,
    'min_samples_split': 10,
    'random_state': 42
}

gbr_lower = GradientBoostingRegressor(loss='quantile', alpha=0.05, **other_params)
gbr_upper = GradientBoostingRegressor(loss='quantile', alpha=0.95, **other_params)

gbr_lower.fit(X_train, y_train)
gbr_upper.fit(X_train, y_train)

pred_lower = gbr_lower.predict(X_val)
pred_upper = gbr_upper.predict(X_val)

# 评估 W_alpha
score = interval_score(y_val, pred_lower, pred_upper, alpha=0.1)
print(f"W_alpha score: {score:.5f}")

In [ ]:
def best_features(model):
    # 获取特征重要性
    importance = model.feature_importances_
    features = X_train.columns
    
    # 打包成 DataFrame
    feat_imp = pd.DataFrame({
        'feature': features,
        'importance': importance
    }).sort_values('importance', ascending=False)
    
    # 显示前 30 个最重要的特征
    print(feat_imp.head(30))


best_features(model_lower)
best_features(model_upper)


In [ ]:
[k for k in X.columns if 'knn_price' in k]

In [ ]:
def add_knn_price_features(df, lat_col='latitude', lon_col='longitude', target_col='target', ks=[5, 10, 20]):
    coords = df[[lat_col, lon_col]].values
    target = df[target_col].values
    tree = KDTree(coords, metric='euclidean')

    # 用于保存所有新特征
    knn_features = {}

    for k in ks:
        # 找最近邻（包括自己）
        dists, indices = tree.query(coords, k=k+1)  # k+1 是因为自己也算在内

        # 排除自身
        neighbor_targets = np.array([target[idxs[1:]] for idxs in indices])

        # 聚合统计
        knn_features[f'knn_price_mean_{k}'] = neighbor_targets.mean(axis=1)
        knn_features[f'knn_price_std_{k}'] = neighbor_targets.std(axis=1)
        knn_features[f'knn_price_range_{k}'] = neighbor_targets.max(axis=1) - neighbor_targets.min(axis=1)

    # 加入原始 df
    for col_name, values in knn_features.items():
        df[col_name] = values

    return df 


In [ ]:
# 假设 df 中已经有 latitude、longitude、target（真实价格）列
df = add_knn_price_features(train, target_col='sale_price', ks=[5, 10, 20])


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_knn_feature(df, feature_col='knn_price_mean_5'):
    plt.figure(figsize=(10, 8))
    sc = plt.scatter(df['longitude'], df['latitude'], 
                     c=df[feature_col], cmap='viridis', s=5)
    plt.colorbar(sc, label=feature_col)
    plt.title(f'Spatial distribution of {feature_col}')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.grid(True)
    plt.show()


In [ ]:
plot_knn_feature(df, 'knn_price_mean_20')
plot_knn_feature(df, 'knn_price_mean_10')
plot_knn_feature(df, 'knn_price_mean_5')

In [ ]:

df = train.copy()
addresses = (df['city'].fillna('') + ' ' + df['subdivision'].fillna('')).str.lower()

# 分词（空格切）
# 用 CountVectorizer 可以保留频率稀疏性

vectorizer = CountVectorizer(
    max_features=1000,         # 可调大小
    stop_words=None,      # 去常见词
    token_pattern=r'\b\w+\b',  # 标准单词
    ngram_range=(1, 2)         # 一元和二元组都试试
)
address_vecs = vectorizer.fit_transform(addresses)

# 得到 token -> index 映射
tokens = vectorizer.get_feature_names_out()
# 每个 token 的总出现频率（在所有样本中出现的总次数）
token_freq = np.asarray(address_vecs.sum(axis=0)).ravel()

# 排序
sorted_idx = np.argsort(-token_freq)
top_tokens = [(tokens[i], token_freq[i]) for i in sorted_idx[:200]]

# 展示前 50 个 token 和它们的频率
for t, f in top_tokens:
    print(f"{t}: {f}")
svd = TruncatedSVD(n_components=100, random_state=42)
address_pca = svd.fit_transform(address_vecs)

# 累计解释方差
explained = np.cumsum(svd.explained_variance_ratio_)
plt.plot(range(1, len(explained) + 1), explained)
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('Truncated SVD on Address')
plt.grid(True)
plt.show()


In [ ]:
待处理特征：
改造时间距离现在多少年
地址经纬度做近邻，最好找出价格最高的中心点
尝试销售年份和submarke捆绑，销售月份和地区捆绑
PCA提取地址主成分


In [27]:
def winkler_score(y_true, lower, upper, alpha=0.1, return_coverage=False):

    y_true = np.asarray(y_true)
    lower = np.asarray(lower)
    upper = np.asarray(upper)

    width = upper - lower
    penalty_lower = 2 / alpha * (lower - y_true)
    penalty_upper = 2 / alpha * (y_true - upper)

    score = width.copy()
    score += np.where(y_true < lower, penalty_lower, 0)
    score += np.where(y_true > upper, penalty_upper, 0)

    if return_coverage:
        inside = (y_true >= lower) & (y_true <= upper)
        coverage = np.mean(inside)
        return np.mean(score), coverage

    return np.mean(score)

In [34]:

import numpy as np
import pandas as pd
from quantile_forest import RandomForestQuantileRegressor
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder

random_state = 0
np.random.seed(random_state)

qrf = RandomForestQuantileRegressor(
    n_estimators=300,
    max_features=0.333,
    max_samples_leaf=10,
    random_state=random_state,
)
qrf.fit(X_train, y_train)

,n_estimators,300
,default_quantiles,0.5
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,max_samples_leaf,10
,min_weight_fraction_leaf,0.0
,max_features,0.333
,max_leaf_nodes,None
,min_impurity_decrease,0.0


In [35]:
alpha = 0.1
quantiles = [alpha / 2, 1 - alpha / 2]

y_val_pred = qrf.predict(X_val, quantiles=quantiles)
y_val_pred = pd.DataFrame(y_val_pred, columns=["pi_lower", "pi_upper"])
mws, coverage = winkler_score(
    y_val,
    y_val_pred["pi_lower"],
    y_val_pred["pi_upper"],
    alpha=alpha,
    return_coverage=True,
)

print("Mean Winkler Score:", round(mws, 2))
print("Coverage:", round(coverage * 100, 1), "%")

Mean Winkler Score: 0.87
Coverage: 90.2 %


In [1]:
print("Mean Winkler Score:", round(mws, 2))
print("Coverage:", round(coverage * 100, 1), "%")

NameError: name 'mws' is not defined

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 模型是 qrf
feature_importances = qrf.feature_importances_
features = X_train.columns  # 或你使用的特征列名

importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': feature_importances
}).sort_values(by='Importance', ascending=False)

# 打印前十个最重要的特征
print(importance_df.head(10))

# 可视化
importance_df.head(10).plot(kind='barh', x='Feature', y='Importance', legend=False)
plt.gca().invert_yaxis()
plt.title("Top 10 Important Features")
plt.show()


In [26]:
import numpy as np
import pandas as pd

from datetime import datetime
from scipy.stats import skew 
from scipy.special import boxcox1p
from scipy.stats import boxcox_normmax
from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error
from mlxtend.regressor import StackingCVRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import matplotlib.pyplot as plt
import scipy.stats as stats
import sklearn.linear_model as linear_model
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.neighbors import KDTree
from sklearn.decomposition import TruncatedSVD
from sklearn.neighbors import BallTree
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import category_encoders as ce


import os
print(os.listdir())

import warnings
warnings.filterwarnings('ignore')

['house_price_competition_day_2_failed_trying.ipynb', 'house_price_advanced_view.ipynb', 'addr_kmeans.pkl', 'submission.csv', 'house_price_advanced_models.ipynb', 'my_model_submission.csv1', 'my_model_submission4.csv', 'house_price', 'addr_umap.pkl', 'Day1.ipynb', 'titanic', 'house-prices-advanced-regression-techniques.zip', 'titanic.zip', 'my_model_submission3.csv', 'house_price_competition_view_day2.ipynb', 'my_model_submission0.csv', 'X_umap.npy', 'addr_tfidf.pkl', 'Day2 Housing Price.ipynb', 'pca_model.pkl', 'my_model_submission.csv', 'predictions.csv', 'umap_model.pkl', 'predictions4.csv', 'my_model_submission1.csv', 'competition_day7.ipynb', 'house_price_view.ipynb', 'home-data-for-ml-course.zip', 'Untitled2.ipynb', '.ipynb_checkpoints', 'home-data-for-ml-course', 'house_price.ipynb', 'predictions3.csv', 'failed_trying_with_nns_day_9.ipynb', 'not_so_bad_trying_with_lightgbm_day_8.ipynb', 'my_model_submission2.csv', 'best_interval_model.pth', 'predictions2.csv']


In [51]:
# ========== 1. 定义网络结构 ==========
class QuantileRegressor(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.3),

            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.2),

            nn.Linear(256, 128),
            nn.ReLU(),

            nn.Linear(128, 2),  # 可以加入新层

        )

    def forward(self, x):
        return self.net(x)



# ========== 2. Quantile Loss (Pinball Loss) ==========
def pinball_interval_loss(preds, target, alpha=0.1, coverage=1 ,penalty_weight=10.0):
    y_low = preds[:, 0]
    y_high = preds[:, 1]
    y = target.squeeze()

    # 覆盖惩罚
    under = (y < y_low).float()
    over = (y > y_high).float()
    if coverage < 0.9:
        #cover =((1-coverage)*250)
        cover = 1.5

    else: 
        cover = 1

    #cover =((1-coverage)*20) + 1

        
        
    miss_penalty = 2.0 / alpha * ((y_low - y) * under + (y - y_high) * over) * cover

    # 区间宽度惩罚
    width_penalty = (y_high - y_low)

    # 结构惩罚项：low > high 是不合法的
    structure_penalty = torch.mean(torch.relu(y_low - y_high)) * penalty_weight

    return torch.mean(width_penalty + miss_penalty) + structure_penalty


def interval_score_loss(preds, target, alpha=0.1):
    y_low = preds[:, 0]
    y_high = preds[:, 1]
    y = target.squeeze()

    width = y_high - y_low
    below = (y < y_low).float()
    above = (y > y_high).float()

    under_penalty = (2 / alpha) * (y_low - y) * below
    over_penalty = (2 / alpha) * (y - y_high) * above

    score = width + under_penalty + over_penalty
    return torch.mean(score)



# ========== 3. 训练函数 ==========
def train_model(X, y, alpha=0.1, epochs=200, batch_size=1024, lr=1e-3):
    torch.backends.cudnn.benchmark = True
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 数据准备
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, random_state=42)

    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
    X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
    y_val_tensor = torch.tensor(y_val, dtype=torch.float32)

    train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=batch_size, num_workers=4, shuffle=True)
    val_loader = DataLoader(TensorDataset(X_val_tensor, y_val_tensor), batch_size=batch_size)

    # 模型定义
    model = QuantileRegressor(input_dim=X.shape[1]).to(device)

    #optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2, weight_decay=1e-5)
    #scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)


    best_val_loss = float('inf')
    best_model = None
    patience, patience_counter = 10, 0

    # ========== 4. 训练循环 ==========
    coverage = 1
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb)
            loss = pinball_interval_loss(preds, yb, alpha, coverage)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            

        # 验证
        model.eval()
        with torch.no_grad():
            val_loss = 0
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                preds = model(xb)
                loss = pinball_interval_loss(preds, yb, alpha, coverage)
                val_loss += loss.item()


        print(f"Epoch {epoch+1}, Train Loss: {total_loss:.4f}, Val Loss: {val_loss:.4f}")

        with torch.no_grad():
            preds = model(X_val_tensor.to(device)).cpu().numpy()
            y_true = y_val_tensor.cpu().numpy()
            coverage = np.mean((y_true >= preds[:, 0]) & (y_true <= preds[:, 1]))
            avg_width = np.mean(preds[:, 1] - preds[:, 0])
            winkler_val_score = winkler_score(y_true, preds[:, 0], preds[:, 1], alpha=0.1)
            print(f"Val Coverage: {coverage:.4f}, Avg Width: {avg_width:.2f}, Winkler Score: {winkler_val_score:.4f}")

        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Current LR: {current_lr:.6f}")

        
        # Early Stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping triggered!")
                break

    model.load_state_dict(best_model)
    torch.save(best_model, 'best_interval_model.pth')
    return model


# ========== 5. 用法示例 ==========
# X_final: numpy 数组 (N, D)
# y_train_lower, y_train_upper: numpy 数组 (N,)
# 合并成 target 矩阵：
# y_train = np.stack([y_train_lower, y_train_upper], axis=1)

# model = train_model(X_final, y_train)

# 输出预测
# model(torch.tensor(X_test).to(device)).cpu().detach().numpy()

In [52]:
X = X_train.astype(np.float32)
y = y_train.astype(np.float32)
X_val = X_val.astype(np.float32)
y_val = y_val.astype(np.float32)
from torch.optim.lr_scheduler import ReduceLROnPlateau
import math

model = train_model(X.values, y.values, epochs=150)
print("Finished training")

Epoch 1, Train Loss: 10708.8559, Val Loss: 58.1188
Val Coverage: 0.9901, Avg Width: 3.54, Winkler Score: 3.6340
Current LR: 0.000337
Epoch 2, Train Loss: 510.1209, Val Loss: 40.4953
Val Coverage: 0.9982, Avg Width: 2.52, Winkler Score: 2.5311
Current LR: 0.000415
Epoch 3, Train Loss: 446.7573, Val Loss: 38.7701
Val Coverage: 0.9988, Avg Width: 2.42, Winkler Score: 2.4232
Current LR: 0.000422
Epoch 4, Train Loss: 430.5236, Val Loss: 36.4940
Val Coverage: 0.9985, Avg Width: 2.27, Winkler Score: 2.2810
Current LR: 0.000430
Epoch 5, Train Loss: 418.5709, Val Loss: 41.3424
Val Coverage: 0.9995, Avg Width: 2.57, Winkler Score: 2.5844
Current LR: 0.000412
Epoch 6, Train Loss: 411.3625, Val Loss: 34.6488
Val Coverage: 0.9982, Avg Width: 2.15, Winkler Score: 2.1657
Current LR: 0.000437
Epoch 7, Train Loss: 398.0276, Val Loss: 31.0694
Val Coverage: 0.9976, Avg Width: 1.93, Winkler Score: 1.9419
Current LR: 0.000449
Epoch 8, Train Loss: 397.0057, Val Loss: 35.2183
Val Coverage: 0.9982, Avg Width:

In [83]:
X_val = X_val.astype(np.float32)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval()
model.load_state_dict(torch.load('best_interval_model.pth'))
pred = model(torch.tensor(X_val.values).to(device)).cpu().detach().numpy()
print("Finished prediction")
pred_low = pred[:,0]
pred_high = pred[:,1]
score, coverage = winkler_score(y_val, pred_low, pred_high, 0.1, True)
print(score, coverage)

Finished prediction
4.3560014 0.8791


In [86]:
score, coverage = winkler_score(y_val, pred_lower, pred_upper, 0.1, True)
print(score, coverage)

0.9108374315797668 0.74895


In [67]:
pred_low = pred_low_fixed
pred_high = pred_high_fixed
print("y_val range:", y_val.min(), y_val.max())
print("pred_low range:", pred_low.min(), pred_low.max())
print("pred_high range:", pred_high.min(), pred_high.max())

print("y_val mean/std:", y_val.mean(), y_val.std())
print("pred_low mean/std:", pred_low.mean(), pred_low.std())
print("pred_high mean/std:", pred_high.mean(), pred_high.std())


y_val range: 10.915107 14.913456
pred_low range: 2.3213034 7.309837
pred_high range: 2.5437503 7.5895324
y_val mean/std: 13.075315 0.6185022
pred_low mean/std: 2.6174982 0.06950945
pred_high mean/std: 2.6618912 0.07179182


In [ ]:
y_val, y_train

In [ ]:
X, X_val

In [85]:
print(pred_lower)

[12.20975791 12.97986176 13.35793305 ... 12.71927317 13.52720644
 12.79062808]


In [88]:
print(pred_lower-pred_low)
print(pred_upper-pred_high)

[0.13143454 0.20403054 0.0253296  ... 0.20264681 0.27328111 0.17728579]
[-0.14666523  0.05712242  0.21336406 ... -0.12941224 -0.00259935
 -0.10986916]
